In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from pathlib import Path

# ============================================================
# Read metrics from results folders only (no hardcoded values)
# ============================================================
BASE = Path('.')
results_map = {
    'ProtT5': BASE / 'results_enzyme_ProtT5',
    'ANKH': BASE / 'results_enzyme_Ankh',
    'ESM2': BASE / 'results_enzyme_ESM2',
}

window_html_patterns = [
    re.compile(
        r"Window\s+MLP\s+results:\s*Accuracy\s*([0-9.]+)\s*\|\s*Macro\s*F1\s*([0-9.]+)",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"Active[-\s]*site\s+Window\s+Analysis.*?Accuracy\s*[:=]?\s*([0-9.]+).*?Macro\s*F1\s*[:=]?\s*([0-9.]+)",
        flags=re.IGNORECASE | re.DOTALL,
    ),
]
window_txt_patterns = [
    re.compile(
        r"Window.*?Accuracy\s*[:=]\s*([0-9.]+).*?Macro\s*F1\s*[:=]\s*([0-9.]+)",
        flags=re.IGNORECASE | re.DOTALL,
    ),
]


def _match_metrics(text: str, patterns: list[re.Pattern]):
    for pat in patterns:
        m = pat.search(text)
        if m:
            vals = [g for g in m.groups() if g is not None]
            if len(vals) >= 2:
                return float(vals[0]), float(vals[1])
    return np.nan, np.nan


def extract_window_metrics(results_dir: Path):
    """Extract window Accuracy/Macro-F1 from report files when available."""
    # 1) Preferred: structured CSV (if future pipelines add it)
    win_csv = results_dir / 'active_site_window_metrics.csv'
    if win_csv.exists():
        win_df = pd.read_csv(win_csv)
        if {'accuracy', 'macro_f1'}.issubset(win_df.columns):
            best = win_df.sort_values(['macro_f1', 'accuracy'], ascending=False).iloc[0]
            return float(best['accuracy']), float(best['macro_f1'])

    # 2) HTML report explicit window section
    html_file = results_dir / 'interpretation_report.html'
    if html_file.exists():
        html_text = html_file.read_text(encoding='utf-8', errors='ignore')
        acc, f1 = _match_metrics(html_text, window_html_patterns)
        if np.isfinite(acc) and np.isfinite(f1):
            return acc, f1

    # 3) Text summary explicit window section
    txt_file = results_dir / 'analysis_summary.txt'
    if txt_file.exists():
        txt_text = txt_file.read_text(encoding='utf-8', errors='ignore')
        acc, f1 = _match_metrics(txt_text, window_txt_patterns)
        if np.isfinite(acc) and np.isfinite(f1):
            return acc, f1

    return np.nan, np.nan


rows = []
for model, rdir in results_map.items():
    summary_csv = rdir / 'summary_metrics.csv'
    if not summary_csv.exists():
        raise FileNotFoundError(f'Missing full-length metrics file: {summary_csv}')

    # Full-length: best row by macro_f1 then accuracy
    full_df = pd.read_csv(summary_csv)
    full_best = full_df.sort_values(['macro_f1', 'accuracy'], ascending=False).iloc[0]

    # Active-site window metrics from result files
    win_acc, win_f1 = extract_window_metrics(rdir)

    rows.append({
        'Model': model,
        'Full-length Accuracy': float(full_best['accuracy']),
        'Full-length Macro-F1': float(full_best['macro_f1']),
        'Window Accuracy': win_acc,
        'Window Macro-F1': win_f1,
    })

metrics_df = pd.DataFrame(rows)

print('Extracted metrics from result files:')
print(metrics_df.to_string(index=False))

missing_models = metrics_df[
    metrics_df[['Window Accuracy', 'Window Macro-F1']].isna().any(axis=1)
]['Model'].tolist()
if missing_models:
    print(f"\nWarning: Missing window metrics for: {', '.join(missing_models)}")

# ============================================================
# Plot Style (Publication Ready)
# ============================================================
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
})

models = metrics_df['Model'].tolist()
full_accuracy = metrics_df['Full-length Accuracy'].to_numpy(dtype=float)
full_f1 = metrics_df['Full-length Macro-F1'].to_numpy(dtype=float)
win_accuracy = metrics_df['Window Accuracy'].to_numpy(dtype=float)
win_f1 = metrics_df['Window Macro-F1'].to_numpy(dtype=float)

x = np.arange(len(models))
width = 0.32
colors = {
    'full': '#bdbdbd',
    'win': '#1f77b4',
}

fig, axes = plt.subplots(1, 2, figsize=(10, 5.5), dpi=300)

# ============================================================
# Accuracy panel
# ============================================================
ax = axes[0]
bars1 = ax.bar(
    x - width / 2,
    full_accuracy,
    width,
    label='Full-length',
    color=colors['full'],
    edgecolor='black',
    linewidth=0.8,
)
bars2 = ax.bar(
    x + width / 2,
    win_accuracy,
    width,
    label='15-aa catalytic window',
    color=colors['win'],
    edgecolor='black',
    linewidth=0.8,
)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=15)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Accuracy', fontsize=18)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if np.isfinite(height):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.008,
                f'{height:.2f}',
                ha='center',
                va='bottom',
                fontsize=10,
            )

# ============================================================
# Macro-F1 panel
# ============================================================
ax = axes[1]
bars1 = ax.bar(
    x - width / 2,
    full_f1,
    width,
    label='Full-length',
    color=colors['full'],
    edgecolor='black',
    linewidth=0.8,
)
bars2 = ax.bar(
    x + width / 2,
    win_f1,
    width,
    label='15-aa catalytic window',
    color=colors['win'],
    edgecolor='black',
    linewidth=0.8,
)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=15)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Macro-F1', fontsize=18)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if np.isfinite(height):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.008,
                f'{height:.2f}',
                ha='center',
                va='bottom',
                fontsize=10,
            )

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='lower center',
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
    fontsize=14,
)

fig.suptitle(
    'Performance Comparison: Full-Length Protein vs 15-aa Catalytic Window',
    fontsize=15,
    y=1.02,
)

plt.tight_layout(rect=[0, 0.06, 1, 1])

# ============================================================
# Save outputs in a separate folder
# ============================================================
out_dir = BASE / 'comparison_fullseq_vs_window'
out_dir.mkdir(parents=True, exist_ok=True)

metrics_df['Accuracy Improvement (%)'] = (
    (metrics_df['Window Accuracy'] - metrics_df['Full-length Accuracy']) * 100
).round(2)
metrics_df['F1 Improvement (%)'] = (
    (metrics_df['Window Macro-F1'] - metrics_df['Full-length Macro-F1']) * 100
).round(2)

metrics_df.to_csv(out_dir / 'comparison_summary.csv', index=False)
plt.savefig(out_dir / 'fullseq_vs_window_comparison.png', dpi=300, bbox_inches='tight')

print(f'\nSaved: {out_dir / "comparison_summary.csv"}')
print(f'Saved: {out_dir / "fullseq_vs_window_comparison.png"}')

plt.show()